In [ ]:
%pip install openai

# Workload Profiler — Pay-Per-Token

Profiles **per-request** token and latency characteristics against a PPT endpoint. Use the output as inputs to the ITPM/OTPM calculation and the Databricks GenAI Pricing Calculator.

| Metric | Description | Used for |
|---|---|---|
| **Input tokens** | Prompt tokens per request | `ITPM = input_tokens × QPM` |
| **Output tokens** | Completion tokens per request | `OTPM = output_tokens × QPM` |
| **TTFT** | Time to first token (ms) | Latency baseline |
| **TPOT** | Time per output token (ms) | Latency baseline |

**Note:** This notebook cannot calculate ITPM or OTPM directly — those require QPM, which comes from your production traffic or estimation. See the README for how to derive QPM for your workload type.

In [ ]:
import os

os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Change to the pay-per-token model you want to profile
PPT_MODEL = "databricks-gpt-oss-20b"

# PPT rate limits sourced from:
# https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/limits
# QPH = queries per hour (None = not published). Update as limits change.
PPT_LIMITS = {
    # GPT OSS
    "databricks-gpt-oss-120b":              {"itpm": 200_000, "otpm": 10_000, "qph": None},
    "databricks-gpt-oss-20b":               {"itpm": 200_000, "otpm": 10_000, "qph": None},
    # Meta Llama
    "databricks-meta-llama-4-maverick":     {"itpm": 200_000, "otpm": 10_000, "qph": 2_400},
    "databricks-meta-llama-3-3-70b-instruct": {"itpm": 200_000, "otpm": 10_000, "qph": 2_400},
    "databricks-meta-llama-3-1-8b-instruct":  {"itpm": 200_000, "otpm": 10_000, "qph": 7_200},
    "databricks-meta-llama-3-1-405b-instruct": {"itpm": 5_000,  "otpm": 500,    "qph": 1_200},
    # Google
    "databricks-gemma-3-12b":              {"itpm": 200_000, "otpm": 10_000, "qph": 7_200},
    "databricks-gemini-2-5-pro":           {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    "databricks-gemini-2-5-flash":         {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    # Anthropic Claude
    "databricks-claude-sonnet-4":          {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    "databricks-claude-opus-4-7":          {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    "databricks-claude-haiku-4-5":         {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
}

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints",
    api_key=os.environ["DATABRICKS_TOKEN"],
)

In [ ]:
import time


def profile_request(messages: list[dict]) -> dict:
    """
    Two calls per query:
    1. Non-streaming  → input/output token counts (usage is in the response object)
    2. Streaming      → TTFT and TPOT (Databricks API does not support stream_options)

    TTFT = time from request start → first token received
    TPOT = (total_time - TTFT) / (output_tokens - 1)
    """
    # --- token counts ---
    response = client.chat.completions.create(
        model=PPT_MODEL,
        messages=messages,
    )
    input_tokens  = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    # --- latency metrics ---
    ttft_ms = None
    t_start = time.perf_counter()

    stream = client.chat.completions.create(
        model=PPT_MODEL,
        messages=messages,
        stream=True,
    )

    for chunk in stream:
        if ttft_ms is None and chunk.choices and chunk.choices[0].delta.content:
            ttft_ms = (time.perf_counter() - t_start) * 1000

    total_ms = (time.perf_counter() - t_start) * 1000
    tpot_ms  = (total_ms - ttft_ms) / (output_tokens - 1) if output_tokens > 1 else 0.0

    return {
        "input_tokens":  input_tokens,
        "output_tokens": output_tokens,
        "ttft_ms":       round(ttft_ms, 2),
        "tpot_ms":       round(tpot_ms, 2),
        "total_ms":      round(total_ms, 2),
    }

## Sample queries

Replace `SAMPLE_QUERIES` with prompts that reflect your actual workload. The goal is to capture a realistic distribution of input and output lengths — **the sample queries below will underestimate input tokens for most production workloads**.

**What "representative" means depends on your workload type:**

**RAG / retrieval-augmented pipelines** — a user query alone is not representative. The LLM receives the query *plus* the retrieved chunks assembled into a prompt. Your sample queries must include that assembled context, otherwise profiled input tokens will be far shorter than production. Construct samples by running your retrieval pipeline on a set of real or realistic questions and capturing the full prompt sent to the LLM.

**Chat / Q&A** — include your system prompt and a realistic message history. A single-turn bare question omits both.

**Agentic / tool-calling workflows** — this notebook profiles a single LLM call. Agentic chains make multiple LLM round-trips per user request; each tool call response and result is fed back as additional input tokens on the next turn. For these workloads, either instrument your orchestrator to log token counts across the full chain and sum them, or profile each step individually and add.

In all cases: run enough samples (20–50+) to get a stable average. A handful of queries will produce a noisy estimate.

In [ ]:
# Replace or extend SAMPLE_QUERIES with prompts representative of your workload.
# Each entry is a messages list — the same format passed to the OpenAI chat completions API.
#
# RAG / retrieval pipelines: include the assembled context (system prompt + retrieved chunks
# + user query), not the bare question. The examples below intentionally include a realistic
# context block so you can run this cell as-is and get a meaningful input token estimate.

_rag_context = """\
[Document 1]
Provisioned Throughput (PT) provides dedicated inference capacity for Foundation Model APIs. \
Unlike pay-per-token pricing, PT reserves model units — fixed compute allocations billed \
hourly regardless of usage. Each model unit translates to a guaranteed tokens-per-second \
throughput floor for a specific model. PT endpoints support burst scaling: when traffic \
temporarily exceeds provisioned capacity, Databricks can step up to the next model unit \
tier if regional capacity is available.

[Document 2]
Model units are sized based on your expected query rate and the average input/output token \
distribution of your requests. The Databricks GenAI Pricing Calculator accepts average input \
tokens per request, average output tokens per request, and queries per minute to recommend \
the appropriate number of model units. Output tokens are more compute-intensive than input \
tokens due to the autoregressive nature of decoding, so workloads with long outputs require \
more model units per QPM than workloads with short outputs.

[Document 3]
For production deployments, a common pattern is to pair a PT endpoint for baseline traffic \
with a pay-per-token fallback for overflow. The PT endpoint handles your provisioned QPM \
floor with guaranteed throughput, while the PPT model absorbs bursts that exceed PT capacity. \
This avoids the need to overprovision PT for peak traffic while ensuring requests are served \
during spikes.\
"""

SAMPLE_QUERIES = [
    # RAG / retrieval — assembled prompt with system instruction, retrieved context, and user question.
    # Replace _rag_context with chunks returned by your actual vector DB retrieval.
    [
        {"role": "system", "content": "You are a helpful assistant. Answer the user's question using only the provided context. Be concise."},
        {"role": "user",   "content": f"Context:\n{_rag_context}\n\nQuestion: What is the difference between Provisioned Throughput and Pay-Per-Token, and when should I use each?"},
    ],

    # Simple Q&A — add your system prompt and realistic message history here
    [{"role": "user", "content": "What is machine learning? Explain in 3 paragraphs."}],
    [{"role": "user", "content": "Write a Python function that reads a CSV file and returns the top 5 rows by a given column."}],
]

results = []

for i, messages in enumerate(SAMPLE_QUERIES):
    preview = messages[-1]["content"][:60]
    print(f"[{i+1}/{len(SAMPLE_QUERIES)}] {preview}...")
    result = profile_request(messages)
    results.append({"query": preview, **result})
    print(f"  input={result['input_tokens']} output={result['output_tokens']} "
          f"ttft={result['ttft_ms']}ms tpot={result['tpot_ms']}ms\n")

## Agentic / Tool-Calling Workloads

For agents that make multiple LLM calls per user request (e.g., a RAG agent that calls a vector search tool, gets results, then calls the LLM again to synthesize), the `SAMPLE_QUERIES` approach only captures one call. The true per-request token cost is the sum across all LLM calls in the chain.

MLflow traces capture this automatically. Each LLM call in the chain becomes a `CHAT_MODEL` span; tool executions become `TOOL` spans (no token cost). `trace.info.token_usage` aggregates across all `CHAT_MODEL` spans in the trace, giving you the correct per-request total.

A typical agentic trace looks like:

```
root (AGENT)
 ├── llm_call_1 (CHAT_MODEL)  input: 512   output: 42    ← decide which tool to call
 ├── vector_search  (TOOL)                               ← no token cost
 └── llm_call_2 (CHAT_MODEL)  input: 1,828 output: 145  ← synthesize answer with retrieved context
                                            ─────────────
                               total:       input: 2,340  output: 187
```

If your agent is already instrumented with MLflow tracing, run the cell below instead of (or alongside) `SAMPLE_QUERIES` to populate `results` with per-trace token totals.

In [ ]:
import json
from typing import Optional

# ── Paste your traces here ────────────────────────────────────────────────────
# Retrieve traces from your MLflow experiment and serialize to dicts:
#
#   import mlflow
#   mlflow.set_experiment("/your/experiment/path")
#   raw = mlflow.search_traces(filter_string="status = 'OK'", return_type="list", max_results=50)
#   TRACES = [t.to_dict() for t in raw]
#
# The sample below mirrors the real MLflow trace JSON structure (.to_dict() output)
# so you can verify the extraction is working before swapping in real data.

TRACES = [
    {
        "info": {
            "request_id": "trace-abc123",
            "execution_time_ms": 3820,
            "status": "OK",
            "token_usage": {
                "input_tokens": 2340,   # summed across all CHAT_MODEL spans by MLflow
                "output_tokens": 187,
                "total_tokens": 2527,
            },
        },
        "data": {
            "spans": [
                {
                    "name": "agent",
                    "attributes": {"mlflow.spanType": '"AGENT"'},
                },
                {
                    "name": "llm_call_1",
                    "attributes": {
                        "mlflow.spanType": '"CHAT_MODEL"',
                        "mlflow.chat.tokenUsage": json.dumps(
                            {"input_tokens": 512, "output_tokens": 42, "total_tokens": 554}
                        ),
                    },
                },
                {
                    "name": "vector_search",
                    "attributes": {"mlflow.spanType": '"TOOL"'},  # no token usage on tool spans
                },
                {
                    "name": "llm_call_2",
                    "attributes": {
                        "mlflow.spanType": '"CHAT_MODEL"',
                        "mlflow.chat.tokenUsage": json.dumps(
                            {"input_tokens": 1828, "output_tokens": 145, "total_tokens": 1973}
                        ),
                    },
                },
            ]
        },
    }
]


# ── Extract per-trace token totals ────────────────────────────────────────────
def extract_token_usage(trace: dict) -> Optional[dict]:
    """
    Returns {"input_tokens": int, "output_tokens": int} for a trace dict.
    Uses trace-level token_usage (MLflow aggregates across all CHAT_MODEL spans) when
    available, then falls back to summing CHAT_MODEL spans individually.
    """
    usage = trace.get("info", {}).get("token_usage")
    if usage:
        return {"input_tokens": usage["input_tokens"], "output_tokens": usage["output_tokens"]}

    # Fallback: sum CHAT_MODEL spans
    total_input, total_output = 0, 0
    for span in trace.get("data", {}).get("spans", []):
        attrs = span.get("attributes", {})
        if json.loads(attrs.get("mlflow.spanType", '"UNKNOWN"')) != "CHAT_MODEL":
            continue
        raw = attrs.get("mlflow.chat.tokenUsage") or attrs.get("llm.token_usage.input_tokens")
        if isinstance(raw, str):
            per_span = json.loads(raw)
            total_input  += per_span.get("input_tokens", 0)
            total_output += per_span.get("output_tokens", 0)

    return {"input_tokens": total_input, "output_tokens": total_output} if total_input else None


if "results" not in dir():
    results = []

for trace in TRACES:
    request_id = trace.get("info", {}).get("request_id", "unknown")
    usage = extract_token_usage(trace)
    if not usage:
        print(f"Skipping {request_id} — no token usage found")
        continue

    print(f"trace={request_id}  input={usage['input_tokens']}  output={usage['output_tokens']}  "
          f"total_ms={trace['info'].get('execution_time_ms', 0)}")

    results.append({
        "query":         request_id,
        "input_tokens":  usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
        "ttft_ms":       None,   # not meaningful for multi-call chains
        "tpot_ms":       None,
        "total_ms":      trace["info"].get("execution_time_ms", 0),
    })

In [ ]:
import statistics

def percentile(data: list[float], p: int) -> float:
    sorted_data = sorted(data)
    idx = int(len(sorted_data) * p / 100)
    return round(sorted_data[min(idx, len(sorted_data) - 1)], 2)

def summarize(key: str) -> dict:
    vals = [r[key] for r in results if r[key] is not None]
    if not vals:
        return None
    return {
        "avg":  round(statistics.mean(vals), 2),
        "p50":  percentile(vals, 50),
        "p75":  percentile(vals, 75),
        "p95":  percentile(vals, 95),
        "max":  round(max(vals), 2),
    }

metrics = ["input_tokens", "output_tokens", "ttft_ms", "tpot_ms"]
labels  = ["Input tokens", "Output tokens", "TTFT (ms)", "TPOT (ms)"]

print(f"{'Metric':<18} {'avg':>8} {'p50':>8} {'p75':>8} {'p95':>8} {'max':>8}")
print("-" * 58)
for metric, label in zip(metrics, labels):
    s = summarize(metric)
    if s is None:
        continue   # omit latency rows when results come from traces (not applicable)
    print(f"{label:<18} {s['avg']:>8} {s['p50']:>8} {s['p75']:>8} {s['p95']:>8} {s['max']:>8}")

print(f"\nn={len(results)} requests")
print("\nUse avg input/output tokens and your QPM in the Databricks GenAI Pricing Calculator to size your PT endpoint.")

## PPT Capacity Ceiling for This Workload

If you don't have historical QPM data, you can reverse-calculate the maximum QPM PPT can serve
given your measured request shape. This tells you whether PPT is sufficient before you ever see
production traffic.

```
max_qpm_from_itpm = PPT_ITPM_CEILING / avg_input_tokens
max_qpm_from_otpm = PPT_OTPM_CEILING / avg_output_tokens
ppt_max_qpm       = min(max_qpm_from_itpm, max_qpm_from_otpm)   ← binding constraint
```

If your expected QPM will exceed `ppt_max_qpm`, PT is required.

In [ ]:
import statistics

avg_input  = statistics.mean(r["input_tokens"]  for r in results)
avg_output = statistics.mean(r["output_tokens"] for r in results)

limits = PPT_LIMITS.get(PPT_MODEL)
if limits is None:
    print(f"Warning: {PPT_MODEL} not found in PPT_LIMITS — add it to the config cell.")
else:
    itpm = limits["itpm"]
    otpm = limits["otpm"]
    qph  = limits["qph"]

    constraints = {
        "ITPM": itpm / avg_input,
        "OTPM": otpm / avg_output,
    }
    if qph:
        constraints["QPH"] = qph / 60

    binding     = min(constraints, key=constraints.get)
    ppt_max_qpm = constraints[binding]

    print(f"Avg input tokens/request  : {avg_input:.1f}")
    print(f"Avg output tokens/request : {avg_output:.1f}")
    print()
    print(f"  ITPM ceiling ({itpm:>7,}) → {constraints['ITPM']:.1f} QPM")
    print(f"  OTPM ceiling ({otpm:>7,}) → {constraints['OTPM']:.1f} QPM")
    if qph:
        print(f"  QPH  ceiling ({qph:>7,}) → {constraints['QPH']:.1f} QPM  ({qph:,} QPH)")
    print()
    print(f"  Binding constraint : {binding}")
    print(f"  PPT max QPM        : {ppt_max_qpm:.1f} QPM  ({ppt_max_qpm / 60:.2f} QPS)")
    print()
    print("→ If your expected QPM exceeds this, PPT cannot serve your load. Move to PT.")